# Etapa 1: Selección y caracterización del dataset NYC TLC Yellow Taxi

En esta etapa se caracteriza el dataset de viajes en taxi amarillo de la ciudad de Nueva York durante los años 2024 y 2025, publicado por la New York City Taxi and Limousine Commission (TLC). El dataset contiene registros transaccionales de viajes con atributos de fechas, distancias, tarifas y zonas de origen y destino, lo cual lo hace adecuado para el análisis con PySpark en un entorno de Big Data.

Se utilizan dos años completos (2024 y 2025) para garantizar un volumen total superior a 1 GB y para habilitar análisis comparativo interanual. El año 2026, parcialmente publicado al momento de este trabajo, se reserva como conjunto de validación temporal para las etapas posteriores del proyecto.

El notebook está pensado para ejecutarse de forma portable, tanto localmente como en Google Colab. Todas las rutas de datos son relativas al notebook (`./data/raw`), por lo que cualquier integrante del equipo puede clonar el repositorio y ejecutar las celdas sin ajustes adicionales.

## 0. Configuración del entorno

Este notebook usa las siguientes librerías de Python:

- **pyspark**: motor de cómputo distribuido.
- **findspark**: bootstrap de la sesión local de PySpark.
- **pandas**: se utiliza únicamente para mostrar tablas pequeñas ya agregadas (resumen estadístico, conteo de nulos, matriz de correlación). En este notebook nunca se invoca `.toPandas()` sobre el DataFrame completo de viajes.

La siguiente celda instala las tres librerías. Es idempotente: si ya están instaladas (por ejemplo en `env-pyspark`), pip las reportará como *already satisfied*. En Google Colab se instalan desde cero.

Adicionalmente, si se ejecuta en Google Colab, hay que instalar Java (necesario para correr la JVM de Spark). La celda correspondiente está comentada al final de esta sección y solo se ejecuta en Colab.

In [1]:
# Dependencias de Python para el notebook. Idempotente.
!pip install -q pyspark findspark pandas

In [2]:
# Solo en Google Colab: descomentar para instalar Java (la JVM que ejecuta Spark).
# Localmente con env-pyspark esta línea no es necesaria.
# !apt-get install openjdk-8-jdk-headless -qq > /dev/null

## 1. Descarga de datos

Los archivos se obtienen directamente del CDN oficial del TLC en formato Parquet. Desde 2022 el TLC distribuye los registros de viajes en Parquet de manera nativa, por su mejor compresión y lectura columnar respecto a CSV. La descarga se realiza de forma idempotente: si el archivo ya existe en disco, se omite, lo cual permite re-ejecutar el notebook sin volver a bajar los datos.

In [3]:
from pathlib import Path
import subprocess

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
DATA_DIR = Path("data/raw")
YEARS = [2024, 2025]
LOOKUP_FILE = "taxi_zone_lookup.csv"

In [4]:
def download_if_missing(download_url, target_path):
    """Descarga `download_url` a `target_path` solo si `target_path` no existe.

    Devuelve un string con el estado: 'skip', 'ok' o 'error: <mensaje>'.
    """
    if target_path.exists():
        return "skip"

    target_path.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["curl", "-sSL", "-o", str(target_path), download_url],
        capture_output=True,
        timeout=900,
    )

    if result.returncode != 0:
        return f"error: curl exit {result.returncode}"

    return "ok"

In [5]:
# Parquets mensuales de viajes
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        url = f"{CDN_BASE}/trip-data/{filename}"
        target = DATA_DIR / filename
        status = download_if_missing(url, target)
        print(f"{status:>6}  {filename}")

# Tabla de referencia de zonas de taxi
url = f"{CDN_BASE}/misc/{LOOKUP_FILE}"
target = DATA_DIR / LOOKUP_FILE
status = download_if_missing(url, target)
print(f"{status:>6}  {LOOKUP_FILE}")

  skip  yellow_tripdata_2024-01.parquet
  skip  yellow_tripdata_2024-02.parquet
  skip  yellow_tripdata_2024-03.parquet
  skip  yellow_tripdata_2024-04.parquet
  skip  yellow_tripdata_2024-05.parquet
  skip  yellow_tripdata_2024-06.parquet
  skip  yellow_tripdata_2024-07.parquet
  skip  yellow_tripdata_2024-08.parquet
  skip  yellow_tripdata_2024-09.parquet
  skip  yellow_tripdata_2024-10.parquet
  skip  yellow_tripdata_2024-11.parquet
  skip  yellow_tripdata_2024-12.parquet
  skip  yellow_tripdata_2025-01.parquet
  skip  yellow_tripdata_2025-02.parquet
  skip  yellow_tripdata_2025-03.parquet
  skip  yellow_tripdata_2025-04.parquet
  skip  yellow_tripdata_2025-05.parquet
  skip  yellow_tripdata_2025-06.parquet
  skip  yellow_tripdata_2025-07.parquet
  skip  yellow_tripdata_2025-08.parquet
  skip  yellow_tripdata_2025-09.parquet
  skip  yellow_tripdata_2025-10.parquet
  skip  yellow_tripdata_2025-11.parquet
  skip  yellow_tripdata_2025-12.parquet
  skip  taxi_zone_lookup.csv


## 2. Resumen de archivos descargados

Se reporta el inventario y el tamaño en disco. La rúbrica del curso pide un dataset por encima de 1 GB. El formato Parquet ya está comprimido, por lo que el tamaño en disco es menor al equivalente en CSV pero conserva la totalidad de los registros.

In [6]:
files = sorted(DATA_DIR.glob("*"))
total_bytes = sum(f.stat().st_size for f in files)

for f in files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{size_mb:>8.1f} MB   {f.name}")

print()
print(f"Archivos: {len(files)} (esperados: {len(YEARS) * 12 + 1})")
print(f"Tamaño total: {total_bytes / (1024 ** 3):.2f} GB")

parquets = [f for f in files if f.suffix == ".parquet"]
if parquets:
    avg_mb = sum(f.stat().st_size for f in parquets) / len(parquets) / (1024 ** 2)
print(f"Tamaño promedio por Parquet: {avg_mb:.1f} MB")

     0.0 MB   taxi_zone_lookup.csv
    47.6 MB   yellow_tripdata_2024-01.parquet
    48.0 MB   yellow_tripdata_2024-02.parquet
    57.3 MB   yellow_tripdata_2024-03.parquet
    56.4 MB   yellow_tripdata_2024-04.parquet
    59.7 MB   yellow_tripdata_2024-05.parquet
    57.1 MB   yellow_tripdata_2024-06.parquet
    49.9 MB   yellow_tripdata_2024-07.parquet
    48.7 MB   yellow_tripdata_2024-08.parquet
    58.3 MB   yellow_tripdata_2024-09.parquet
    61.4 MB   yellow_tripdata_2024-10.parquet
    57.8 MB   yellow_tripdata_2024-11.parquet
    58.7 MB   yellow_tripdata_2024-12.parquet
    56.4 MB   yellow_tripdata_2025-01.parquet
    57.5 MB   yellow_tripdata_2025-02.parquet
    66.7 MB   yellow_tripdata_2025-03.parquet
    64.2 MB   yellow_tripdata_2025-04.parquet
    74.2 MB   yellow_tripdata_2025-05.parquet
    70.1 MB   yellow_tripdata_2025-06.parquet
    63.8 MB   yellow_tripdata_2025-07.parquet
    59.4 MB   yellow_tripdata_2025-08.parquet
    69.1 MB   yellow_tripdata_2025-09.parquet

## 3. Carga del dataset con PySpark

En esta sección se inicializa una sesión local de PySpark y se cargan los 24 archivos Parquet de viajes en un único DataFrame, pasando la lista explícita de rutas a `spark.read.parquet()`.

PySpark implementa evaluación diferida (lazy evaluation): las transformaciones se registran pero no se ejecutan hasta que se invoca una acción como `count()` o `show()`. Esto permite optimizar el plan de ejecución y distribuir el trabajo entre particiones. La carga real de datos ocurre cuando se ejecuta la primera acción, no cuando se define el DataFrame.

In [7]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()

# Sube el umbral del log de planes (default 25). Algunas agregaciones de la sección 5
# generan más de 25 expresiones (14 columnas con min y max = 28), lo que dispara un
# WARN benigno que solo trunca el plan en el log, no la computación.
spark.conf.set("spark.sql.debug.maxToStringFields", 100)

print(f"Spark versión: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 02:29:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/03 02:29:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark versión: 4.1.1


### Lista explícita de archivos y unificación de esquemas

En lugar de pasar un patrón glob (`yellow_tripdata_*.parquet`), construimos la lista explícita de rutas con `Path.glob()` y la entregamos a `spark.read.parquet(*paths)`. Esto evita un warning interno que dispara el patrón glob al verificar la existencia de un directorio de metadatos de structured streaming, y deja el output del notebook limpio para el entregable.

Adicionalmente se habilita la opción `mergeSchema=True`. NYC TLC introdujo la columna `cbd_congestion_fee` a partir del 5 de enero de 2025 (cargo por la zona de descongestión central de Manhattan). Los archivos de 2024 no la incluyen, los de 2025 sí. Sin esta opción, Spark toma el esquema del primer archivo y descarta silenciosamente columnas que aparecen en archivos posteriores. Con `mergeSchema=True`, Spark inspecciona el esquema de todos los archivos y construye la unión: los registros de 2024 quedan con `cbd_congestion_fee = null` y los de 2025 con su valor real. Referencia: https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#schema-merging

In [8]:
# Lista explícita de rutas a los 24 archivos mensuales
parquet_paths = sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))

# mergeSchema=True para preservar cbd_congestion_fee (presente solo desde 2025-01-05)
df = spark.read.option("mergeSchema", "true").parquet(*parquet_paths)

print(f"Archivos cargados: {len(parquet_paths)}")

Archivos cargados: 24


In [9]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



### Introspección con printSchema()

El método `printSchema()` muestra el árbol completo de tipos del DataFrame, incluyendo:
- Nombre de cada columna
- Tipo de datos (StringType, LongType, DoubleType, TimestampType, etc.)
- Indicador de nullabilidad: `true` si la columna puede contener nulos, `false` si no.

Esto es más informativo que acceder a `df.columns` o `df.dtypes` directamente. Referencia: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.printSchema.html

In [10]:
n_rows = df.count()

print(f"Total de filas: {n_rows:,}")

Total de filas: 89,892,322


### Acciones y evaluación diferida

`count()` es una acción de Spark que dispara la evaluación completa del plan de ejecución. Durante la primera ejecución, Spark lee todos los archivos Parquet desde disco (aproximadamente 1.4 GB) y cuenta las filas. Este proceso puede tardar varios minutos en una máquina local. El resultado se imprime con formato de separador de miles para mejor legibilidad.

Spark **no** memoriza resultados entre acciones por defecto: si se ejecuta otra acción sobre `df` (otro `count()`, un `summary()`, un `groupBy().agg()`, etc.), Spark vuelve a leer los archivos Parquet desde disco.

Por su parte, `df.rdd.getNumPartitions()` informa cuántas particiones físicas distribuyen el DataFrame. Este número depende del tamaño y cantidad de archivos Parquet leídos, y determina el grado de paralelismo de las operaciones posteriores.

In [11]:
print(f"Particiones: {df.rdd.getNumPartitions()}")
print()

zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv"))
)
print("Tabla de referencia de zonas de taxi:")
zones.show(5)

Particiones: 22

Tabla de referencia de zonas de taxi:
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


### 3.1 Caracterización del catálogo de zonas

Aunque el archivo `taxi_zone_lookup.csv` pesa solo ~12 KB y no califica como Big Data por sí mismo, es la tabla de dimensiones que da semántica a `PULocationID` y `DOLocationID` en los 90 millones de viajes. Antes de avanzar conviene caracterizarlo: tamaño exacto, esquema (re-leído con `inferSchema=True` para que `LocationID` sea entero y no string), distribución por borough y nivel de servicio, presencia de nulos, y cobertura del rango esperado de IDs.

In [12]:
from pyspark.sql import functions as F

# 3.1 Caracterización del catálogo de zonas
print(f"Total de zonas: {zones.count()}")
print()

print("Esquema (con inferSchema):")
zones.printSchema()

print("Distribución por Borough:")
zones.groupBy("Borough").count().orderBy(F.desc("count")).show(truncate=False)

print("Distribución por service_zone:")
zones.groupBy("service_zone").count().orderBy(F.desc("count")).show(truncate=False)

print("Conteo de nulos por columna:")
zones.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in zones.columns]).show()

print("Cobertura del rango de IDs (esperado 1 a 265):")
ids_present = {row.LocationID for row in zones.select("LocationID").collect()}
expected = set(range(1, 266))
missing = sorted(expected - ids_present)
extra = sorted(ids_present - expected)
print(f"  IDs ausentes en el catálogo: {missing if missing else 'ninguno'}")
print(f"  IDs fuera del rango 1-265:   {extra if extra else 'ninguno'}")

Total de zonas: 265

Esquema (con inferSchema):
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

Distribución por Borough:
+-------------+-----+
|Borough      |count|
+-------------+-----+
|Queens       |69   |
|Manhattan    |69   |
|Brooklyn     |61   |
|Bronx        |43   |
|Staten Island|20   |
|EWR          |1    |
|Unknown      |1    |
|N/A          |1    |
+-------------+-----+

Distribución por service_zone:
+------------+-----+
|service_zone|count|
+------------+-----+
|Boro Zone   |205  |
|Yellow Zone |55   |
|N/A         |2    |
|Airports    |2    |
|EWR         |1    |
+------------+-----+

Conteo de nulos por columna:
+----------+-------+----+------------+
|LocationID|Borough|Zone|service_zone|
+----------+-------+----+------------+
|         0|      0|   0|           0|
+----------+-------+----+------------+

Cobertura del rango de IDs (esperado 1 a 265):


### Lectura del bloque 3.1

El catálogo cubre las zonas TLC referenciadas por `PULocationID` y `DOLocationID` en `df`. La columna `LocationID` es la clave primaria (debe ser única, sin nulos) y los IDs deben ir de 1 a 265 según el diccionario, donde 264 y 265 corresponden a "Unknown" y "Outside of NYC". La columna `service_zone` agrupa zonas por nivel de servicio (Yellow Zone, Boro Zone, Airports, EWR), lo cual será útil en etapas posteriores cuando se analicen patrones de tarifa o demanda por categoría operativa.

La tabla es lo bastante pequeña como para inspeccionarla por completo (no es un caso de Big Data por sí sola), pero su integridad es crítica: cualquier `PULocationID` o `DOLocationID` en `df` que no exista en este catálogo es un error referencial que se documentará en la etapa de calidad.

### Datos cargados

Quedan disponibles dos DataFrames distribuidos en la sesión de Spark:

- `df`: registros de viajes en taxi amarillo (2024-2025), tamaño en disco ~1.4 GB y ~89.9 millones de filas.
- `zones`: tabla de referencia de zonas de taxi (caracterizada en la sección 3.1).

A continuación se documenta el diccionario de variables (sección 4), se validan los rangos efectivos para justificar refinamiento de tipos (sección 5), y finalmente se aplican los downcasts en sitio sobre `df` (sección 6).

## 4. Diccionario de variables (NYC TLC)

La siguiente tabla documenta cada columna del dataset según el diccionario oficial publicado por la New York City Taxi and Limousine Commission. Esta documentación es la fuente de verdad para los rangos válidos, valores enumerados y semántica de cada campo, y se usará en la sección siguiente para justificar refinamientos de tipos y reglas de validación de calidad.

Referencia: https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf (versión 18 de marzo de 2025)

| Columna | Tipo en Parquet | Descripción | Rango / valores válidos |
|---|---|---|---|
| `VendorID` | integer | Código del proveedor TPEP que generó el registro | enum: 1=Creative Mobile Technologies, 2=Curb Mobility, 6=Myle Technologies, 7=Helix |
| `tpep_pickup_datetime` | timestamp_ntz | Fecha y hora cuando el taxímetro fue activado | dentro del año del archivo |
| `tpep_dropoff_datetime` | timestamp_ntz | Fecha y hora cuando el taxímetro fue desactivado | posterior a `tpep_pickup_datetime` |
| `passenger_count` | long | Número de pasajeros en el vehículo | entero ≥ 0; típicamente 1 a 6 |
| `trip_distance` | double | Distancia del viaje en millas reportada por el taxímetro | ≥ 0 millas |
| `RatecodeID` | long | Código de tarifa aplicado al final del viaje | enum: 1=Standard, 2=JFK, 3=Newark, 4=Nassau/Westchester, 5=Negotiated, 6=Group ride, 99=Null/unknown |
| `store_and_fwd_flag` | string | Indica si el registro se almacenó en memoria del vehículo antes de transmitirse al servidor | enum: 'Y', 'N' |
| `PULocationID` | integer | Zona TLC donde se inició el viaje | entero, generalmente 1 a 263 (más 264 y 265 para zonas desconocidas) |
| `DOLocationID` | integer | Zona TLC donde se finalizó el viaje | igual que `PULocationID` |
| `payment_type` | long | Método de pago | enum: 0=Flex Fare, 1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided trip |
| `fare_amount` | double | Tarifa por tiempo y distancia calculada por el taxímetro (USD) | ≥ 0 USD |
| `extra` | double | Extras y recargos diversos (USD) | ≥ 0 USD |
| `mta_tax` | double | Impuesto MTA aplicado según la tarifa metered (USD) | ≥ 0 USD |
| `tip_amount` | double | Propina (solo se registra para pagos con tarjeta de crédito; las propinas en efectivo no aparecen) (USD) | ≥ 0 USD |
| `tolls_amount` | double | Suma total de peajes pagados durante el viaje (USD) | ≥ 0 USD |
| `improvement_surcharge` | double | Recargo de mejora aplicado al inicio del viaje (vigente desde 2015) (USD) | ≥ 0 USD |
| `total_amount` | double | Monto total cobrado al pasajero (no incluye propina en efectivo) (USD) | ≥ 0 USD |
| `congestion_surcharge` | double | Recargo por congestión del estado de NY (USD) | ≥ 0 USD |
| `Airport_fee` | double | Cargo solo para abordajes en LaGuardia y JFK (USD) | ≥ 0 USD |
| `cbd_congestion_fee` | double | Cargo por la zona de descongestión central de Manhattan, vigente desde 2025-01-05 (USD) | ≥ 0 USD; null para registros previos a 2025-01-05 |

## 5. Validación previa al refinamiento de tipos

El esquema actual del dataset está sobre-dimensionado en varias columnas: identificadores enumerados (`VendorID`, `RatecodeID`, `payment_type`) están almacenados como `long` (64 bits) cuando sus valores válidos caben en `byte` (8 bits); las zonas (`PULocationID`, `DOLocationID`) están en `integer` (32 bits) cuando caben en `short` (16 bits); y los montos monetarios están en `double` (64 bits) cuando para tarifas de taxi (típicamente menores a USD 1000) la precisión de `float` (32 bits) es suficiente.

Antes de aplicar cualquier downcast es necesario validar empíricamente que los valores reales en los datos caben en el tipo más estrecho. Esta sección ejecuta tres bloques de validación:

1. **Valores distintos en columnas enumeradas** para confirmar que coinciden con el diccionario oficial.
2. **Mínimos y máximos en columnas numéricas** para confirmar que los rangos efectivos caben en los tipos más estrechos propuestos.
3. **Presencia de la columna `cbd_congestion_fee`** y proporción de nulos por año, para verificar que `mergeSchema=True` la rescató correctamente.

Si las validaciones pasan, se aplicarán los casts en la sección 6 con justificación por columna. Si alguna validación falla, se documenta como hallazgo y se decide si tratarlo como dato sucio en la etapa de calidad o ampliar el tipo destino.

In [13]:
# 5.1 Valores distintos en columnas enumeradas

enum_cols = ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag"]

for col_name in enum_cols:
    print(f"\n{col_name}:")
    (df.groupBy(col_name)
        .count()
        .orderBy(col_name)
        .show(truncate=False))


VendorID:
+--------+--------+
|VendorID|count   |
+--------+--------+
|1       |19302791|
|2       |70027349|
|6       |26051   |
|7       |536131  |
+--------+--------+


RatecodeID:
+----------+--------+
|RatecodeID|count   |
+----------+--------+
|NULL      |15703126|
|1         |68944012|
|2         |2704804 |
|3         |280643  |
|4         |217870  |
|5         |770369  |
|6         |127     |
|99        |1271371 |
+----------+--------+


payment_type:
+------------+--------+
|payment_type|count   |
+------------+--------+
|0           |15703126|
|1           |61506159|
|2           |10194433|
|3           |599890  |
|4           |1888707 |
|5           |7       |
+------------+--------+


store_and_fwd_flag:
+------------------+--------+
|store_and_fwd_flag|count   |
+------------------+--------+
|NULL              |15703126|
|N                 |73902809|
|Y                 |286387  |
+------------------+--------+



### Lectura del bloque 5.1

Cada tabla muestra los valores únicos observados en una columna enumerada y su frecuencia. Se compara contra los valores documentados:

- `VendorID`: deben observarse subconjuntos de {1, 2, 6, 7}.
- `RatecodeID`: subconjuntos de {1, 2, 3, 4, 5, 6, 99}.
- `payment_type`: subconjuntos de {0, 1, 2, 3, 4, 5, 6}.
- `store_and_fwd_flag`: subconjuntos de {'Y', 'N'}.

La presencia de un valor `null` se cuenta como una categoría más y se documenta. Valores fuera del catálogo oficial se registrarán como problema de calidad en el paso siguiente.

In [14]:
# 5.2 Rangos efectivos de columnas numéricas
numeric_cols = [
    "passenger_count", "trip_distance",
    "PULocationID", "DOLocationID",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "cbd_congestion_fee",
]

agg_exprs = []
for c in numeric_cols:
    agg_exprs.append(F.min(c).alias(f"{c}__min"))
    agg_exprs.append(F.max(c).alias(f"{c}__max"))

ranges_row = df.agg(*agg_exprs).first().asDict()

print(f"{'Columna':<25} {'min':>20} {'max':>20}")
for c in numeric_cols:
    mn = ranges_row[f"{c}__min"]
    mx = ranges_row[f"{c}__max"]
    print(f"{c:<25} {str(mn):>20} {str(mx):>20}")

Columna                                    min                  max
passenger_count                              0                    9
trip_distance                              0.0            398608.62
PULocationID                                 1                  265
DOLocationID                                 1                  265
fare_amount                            -2261.2            863372.12
extra                                   -17.39                133.6
mta_tax                                 -21.74              5243.38
tip_amount                             -333.33               999.99
tolls_amount                           -148.17              1702.88
improvement_surcharge                     -1.0                  2.5
total_amount                          -2265.45            863380.37
congestion_surcharge                      -2.5                 2.52
Airport_fee                              -1.75                 6.75
cbd_congestion_fee                       -0.75  

### Lectura del bloque 5.2

La tabla muestra los valores mínimo y máximo observados en cada columna numérica del dataset completo. Se compara cada rango contra el rango del tipo destino propuesto:

| Columna | Tipo destino propuesto | Rango del tipo | Verificación |
|---|---|---|---|
| `passenger_count` | `byte` (8 bits) | -128 a 127 | min ≥ 0 y max ≤ 127 |
| `PULocationID`, `DOLocationID` | `short` (16 bits) | -32 768 a 32 767 | min ≥ 1 y max ≤ 32 767 |
| `trip_distance` | `float` (32 bits) | ~7 dígitos significativos | max menor a 1×10⁷ para preservar precisión decimal |
| Montos en USD (`fare_amount`, `tip_amount`, `total_amount`, etc.) | `float` (32 bits) | ~7 dígitos significativos | max menor a 1×10⁷ |

Si algún max excede el rango del tipo destino, ese tipo se descarta para esa columna o se trata como outlier en la etapa de calidad. Los valores negativos en montos o distancias también se registran: pueden ser anulaciones legítimas (con `RatecodeID` o `payment_type` asociados) o ruido a limpiar.

In [15]:
# 5.3 Verificación de cbd_congestion_fee tras mergeSchema
print("Columnas en df:")
print(df.columns)
print()

cbd_present = "cbd_congestion_fee" in df.columns
print(f"cbd_congestion_fee presente en el esquema: {cbd_present}")

if cbd_present:
    null_by_year = (
        df.withColumn("year", F.year("tpep_pickup_datetime"))
          .groupBy("year")
          .agg(
              F.count("*").alias("n_filas"),
              F.sum(F.col("cbd_congestion_fee").isNull().cast("int")).alias("n_nulos_cbd"),
          )
          .orderBy("year")
    )
    null_by_year.show(truncate=False)

Columnas en df:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']

cbd_congestion_fee presente en el esquema: True
+----+--------+-----------+
|year|n_filas |n_nulos_cbd|
+----+--------+-----------+
|2002|11      |11         |
|2007|1       |0          |
|2008|11      |10         |
|2009|24      |18         |
|2023|10      |10         |
|2024|41169685|41169664   |
|2025|48722578|5          |
|2026|2       |2          |
+----+--------+-----------+



### Lectura del bloque 5.3

Se confirma que `mergeSchema=True` rescató la columna `cbd_congestion_fee` y se cuantifica la proporción de nulos por año:

- Para `year = 2024`: se espera `n_nulos_cbd = n_filas` (100% de nulos). La columna no existía en los archivos de 2024 y `mergeSchema` la introduce vacía.
- Para `year = 2025`: se esperan algunos nulos durante los primeros días de enero (la tarifa entró en vigor el 5 de enero) y luego mayoría con valor.
- Cualquier `year` distinto a 2024 o 2025 indica fechas fuera del período del dataset, lo cual es un problema de calidad a registrar en el paso siguiente.

## 6. Refinamiento del esquema con downcast de tipos

Las validaciones de la sección 5 confirmaron que los rangos efectivos del dataset caben en tipos más estrechos que los que NYC TLC usa en el Parquet original. Esta sección aplica los downcasts en una sola transformación con `selectExpr` (siguiendo el patrón usado en clases `Ejemplo2.ipynb`), reasignando `df` en sitio.

El motivo del refinamiento no es solamente cosmético. Aunque el archivo Parquet en disco no se modifica, los tipos que Spark expone en su DataFrame se reflejan directamente en la representación interna en memoria: cada fila ocupa menos bytes cuando se materializa. Esto reduce el costo de las operaciones que mueven datos: shuffles entre etapas, broadcast joins y la presión sobre la heap del executor. Para un dataset de ~90 millones de filas, el ahorro agregado es relevante.

Se reasigna `df` en sitio para evitar tener dos referencias al mismo plan lógico ocupando espacio mental: el `df` post-cast es la única fuente de verdad para las etapas siguientes.

### Plan de casts y ahorro estimado

| Columna | Tipo actual (bytes) | Tipo destino (bytes) | Bytes ahorrados / fila | Justificación |
|---|---|---|---|---|
| `VendorID` | int (4) | tinyint (1) | 3 | Enum oficial {1, 2, 6, 7}; max=7 ≤ 127 |
| `passenger_count` | long (8) | tinyint (1) | 7 | Rango observado [0, 9]; max=9 ≤ 127 |
| `RatecodeID` | long (8) | tinyint (1) | 7 | Enum {1-6, 99}; max=99 ≤ 127 |
| `PULocationID` | int (4) | smallint (2) | 2 | Rango [1, 265]; cabe en short [-32k, 32k] |
| `DOLocationID` | int (4) | smallint (2) | 2 | Igual que `PULocationID` |
| `payment_type` | long (8) | tinyint (1) | 7 | Enum {0-6}; max observado=5 ≤ 127 |
| `trip_distance` | double (8) | float (4) | 4 | Distancias en millas; precisión float (~7 dígitos significativos) suficiente para valores reales (<10⁴ mi). Outliers (398k mi) son bogus y se filtrarán en limpieza |
| `fare_amount` | double (8) | float (4) | 4 | Tarifas típicas <USD 1000; float preserva centavos hasta ~USD 10⁵ |
| `extra` | double (8) | float (4) | 4 | Recargos pequeños (<USD 200) |
| `mta_tax` | double (8) | float (4) | 4 | Valores pequeños |
| `tip_amount` | double (8) | float (4) | 4 | Propinas observadas <USD 1000 |
| `tolls_amount` | double (8) | float (4) | 4 | Peajes observados <USD 2000 |
| `improvement_surcharge` | double (8) | float (4) | 4 | Recargo casi fijo |
| `total_amount` | double (8) | float (4) | 4 | Mismo argumento que `fare_amount` |
| `congestion_surcharge` | double (8) | float (4) | 4 | Recargo casi fijo (<USD 5) |
| `Airport_fee` | double (8) | float (4) | 4 | Cargo fijo (<USD 10) |
| `cbd_congestion_fee` | double (8) | float (4) | 4 | Cargo fijo (<USD 5) |

**Total ahorro: 72 bytes / fila** sobre las columnas tipadas.

Sobre 89,892,322 filas, el ahorro agregado es de aproximadamente **6.5 GB en presión de memoria** al materializar el DataFrame (shuffles, broadcast, plan execution). El tamaño en disco del Parquet original no cambia: el ahorro se realiza en tiempo de ejecución, donde Spark deserializa cada fila a su representación interna (Catalyst InternalRow) y los tipos más estrechos ocupan menos memoria por fila.

**Columnas que no se modifican**:

| Columna | Tipo | Razón |
|---|---|---|
| `tpep_pickup_datetime` | timestamp_ntz | Ya es el tipo correcto (sin zona horaria, según convención NYC TLC) |
| `tpep_dropoff_datetime` | timestamp_ntz | Igual |
| `store_and_fwd_flag` | string | Solo 3 valores ('Y', 'N', null); convertir a boolean perdería la categoría null. Se evaluará en la etapa de limpieza |

In [16]:
# Aplicación de los casts en una sola transformación con selectExpr.
# Patrón inspirado en sem2/classroom/Ejemplo2.ipynb (DDL inline en cada expresión).
# Se reasigna `df` en sitio: la versión refinada reemplaza a la original, no se duplica el dataset.
df = df.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

print("Esquema refinado:")
df.printSchema()

Esquema refinado:
root
 |-- VendorID: byte (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: byte (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: byte (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: short (nullable = true)
 |-- DOLocationID: short (nullable = true)
 |-- payment_type: byte (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = true)
 |-- Airport_fee: float (nullable = true)
 |-- cbd_congestion_fee: float (nullable = true)



### Esquema refinado

`df` queda reasignado en sitio con los tipos definitivos para el resto del análisis. Las próximas etapas (análisis exploratorio, calidad de datos, muestreo y modelado) operan sobre este esquema.

`df` apunta ahora a un plan lógico que incluye la lectura del Parquet original más las transformaciones de cast. Spark sigue leyendo los archivos en su tipo nativo desde disco y aplica los downcasts en memoria en cada acción.

## 7. Análisis exploratorio descriptivo

Esta sección calcula estadísticas descriptivas distribuidas sobre `df`: resumen numérico, conteo de nulos por columna, top-10 de zonas de origen y destino, patrones temporales (hora, día de semana, mes) y matriz de correlación entre variables numéricas. Cada subsección lleva una explicación del cómputo y una lectura breve del resultado.

### 7.1 Resumen estadístico de variables numéricas

Esta subsección describe la distribución de cada variable continua del dataset. Se separa el cómputo en dos partes para mantener acotado el costo en memoria:

1. `df.summary("count", "mean", "stddev", "min", "max")` calcula los estadísticos básicos en una sola pasada. Se pasan los nombres de los estadísticos de forma explícita para omitir los percentiles del cálculo: con 12 columnas y 90 millones de filas, la versión con percentiles por defecto requiere materializar muchas estructuras intermedias y satura la heap por defecto de la JVM.
2. `approxQuantile(col, probabilities, relativeError)` calcula percentiles columna por columna, usando el algoritmo de Greenwald-Khanna en una sola pasada. Con `relativeError=0.01` el error garantizado es menor al 1% del rango total, suficiente para describir la distribución de tarifas y distancias. Esto se ejecuta en la celda siguiente sobre las cuatro variables más relevantes (`fare_amount`, `trip_distance`, `tip_amount`, `total_amount`) con percentiles 25, 50, 75, 90, 95 y 99.

Para Big Data, `approxQuantile` es preferible a ordenar el DataFrame completo (que requeriría un shuffle global de todos los datos), y es la opción práctica cuando el dataset no cabe en la memoria del driver.

In [17]:
# Columnas continuas de interés para el resumen estadístico.
# Se excluyen columnas categóricas codificadas como numéricas
# (VendorID, RatecodeID, payment_type, PULocationID, DOLocationID)
# y columnas de timestamp, ya que los estadísticos de media/stddev
# no son interpretables para esos tipos.
continuous_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
]

# summary() con stats explícitos: omitimos percentiles porque su cálculo
# sobre 12 columnas y 90M filas requiere ordenar/aproximar todas a la vez
# y satura la heap por defecto. Los percentiles se calculan en la siguiente
# celda con approxQuantile, columna por columna, que es mucho más barato.
summary_sdf = df.select(continuous_cols).summary("count", "mean", "stddev", "min", "max")

# Transponemos para que cada variable quede en su propia fila y los
# estadísticos como columnas. La tabla resultante tiene 12 filas por 5
# columnas, totalmente apto para .toPandas().
summary_pd = summary_sdf.toPandas().set_index("summary").T
summary_pd.index.name = "variable"

print("Resumen estadístico (transpuesto, una fila por variable):")
print(summary_pd.to_string())

Resumen estadístico (transpuesto, una fila por variable):
summary                   count                 mean               stddev       min        max
variable                                                                                      
passenger_count        74189196   1.3147025612732075   0.7739282157205499         0          9
trip_distance          89892322    5.987921635632805    555.9173851671442       0.0  398608.62
fare_amount            89892322   18.808269948301515   117.28427802441267   -2261.2   863372.1
extra                  89892322   1.2658486959542317   1.8225766495787001    -17.39      133.6
mta_tax                89892322   0.4786429412723655    0.569083689401905    -21.74    5243.38
tip_amount             89892322    3.055505823537555    4.030595787460732   -333.33     999.99
tolls_amount           89892322   0.5275469803959015   2.1767747851875714   -148.17    1702.88
improvement_surcharge  89892322   0.9561168077330069   0.2706889786074781      -1.0    

In [18]:
# Percentiles extendidos para las variables clave de tarifa y distancia.
# approxQuantile devuelve una lista de valores por columna.
# relativeError=0.01 garantiza un error menor al 1% del rango de cada variable.
quantile_cols = ["fare_amount", "trip_distance", "tip_amount", "total_amount"]
probabilities = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
rel_error = 0.01

results = df.approxQuantile(quantile_cols, probabilities, rel_error)

print(f"approxQuantile (relativeError={rel_error})")
print(f"{'Percentil':<12}" + "".join(f"{c:>20}" for c in quantile_cols))
for i, p in enumerate(probabilities):
    row = "".join(f"{results[j][i]:>20.2f}" for j in range(len(quantile_cols)))
    print(f"p{int(p*100):<11}" + row)

approxQuantile (relativeError=0.01)
Percentil            fare_amount       trip_distance          tip_amount        total_amount
p25                         9.30                1.02                0.00               15.70
p50                        14.20                1.80                2.38               21.06
p75                        22.60                3.49                4.07               30.45
p90                        39.40                8.55                6.70               52.00
p95                        58.30               12.67               10.93               80.59
p99                    863372.12           398608.62              999.99           863380.38


#### Interpretación del resumen estadístico

Las distribuciones observadas confirman patrones consistentes con datos transaccionales urbanos y a la vez evidencian la presencia de outliers extremos que se procesarán en la etapa de limpieza:

- **`fare_amount`**: media USD 18.81, mediana USD 14.20 (p50 de approxQuantile). El sesgo positivo es marcado: la media supera la mediana en aproximadamente 32%. La desviación estándar (USD 117.28) es enorme respecto al rango intercuartílico (USD 13.30 entre p25 y p75), señal de que la varianza está dominada por valores extremos. El máximo observado (USD 863,372) y el mínimo (USD -2,261) están fuera de cualquier rango realista para un viaje de taxi.
- **`trip_distance`**: media 5.99 millas, mediana 1.80 millas (sesgo positivo extremo, media equivalente a 3.3 veces la mediana). El p99 es 12.67 millas, pero el máximo es 398,608.62 millas, físicamente imposible (más de 16 vueltas a la circunferencia terrestre). La desviación estándar (555.92) es ~93 veces la media, dominada por unos pocos outliers absurdos.
- **`tip_amount`**: el p25 es 0.00, lo que confirma que al menos un cuarto de los registros tienen propina nula (consistente con pagos en efectivo, que NYC TLC no captura). La mediana es USD 2.38 y la media USD 3.06; el p99 llega a USD 10.93 y el máximo a USD 999.99 (probable tope técnico del sistema).
- **`total_amount`**: hereda los outliers de `fare_amount`. Mediana USD 21.06, máximo USD 863,380 (apenas USD 8 sobre el máximo de `fare_amount`, consistente con la suma de recargos casi fijos sobre la tarifa máxima).
- **`passenger_count`**, **`congestion_surcharge`** y **`Airport_fee`**: el `count` reportado para estas columnas es 74,189,196, menor al total del dataset (89,892,322). La diferencia (15,703,126 filas) corresponde al patrón de nulos del bloque Flex Fare identificado en la sección 5.1.
- **`cbd_congestion_fee`**: solo 48,722,602 valores no nulos (54% del dataset); el resto son los registros previos al 5 de enero de 2025 más los Flex Fare de 2025 sin reporte. La media es USD 0.53 y el max USD 1.75, consistente con un cargo casi fijo.

Los valores mínimos negativos en montos monetarios (`fare_amount`, `total_amount`, `tip_amount`, `tolls_amount`, etc.) corresponden a transacciones anuladas o ajustes de crédito legítimos, no a errores. La etapa de limpieza decidirá si se mantienen, se filtran o se etiquetan como reversiones según el `payment_type` o `RatecodeID` asociados.

### 7.2 Conteo de nulos por columna

Una forma natural de contar nulos sería `df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])`: agregar todas las expresiones en una sola pasada. Sobre 21 columnas y 90 millones de filas, sin embargo, esa estrategia bajo la heap por defecto de la JVM saturaba la memoria intermitentemente. Un experimento posterior con `df.agg(*[F.count(c) for c in df.columns])` produjo el mismo resultado: 20 agregaciones simultáneas exceden la memoria disponible.

Como solución pragmática se separa el conteo en dos pasos:

1. `df.summary("count")`: la primitiva ya validada en la sección 7.1, que cuenta valores no nulos por columna pero omite tipos `timestamp_ntz`.
2. `df.agg(*[F.count(c) for c in ts_cols])`: una agregación adicional restringida a las dos columnas de timestamp que summary deja fuera. Como solo procesa 2 columnas, es muy ligera.

Los nulos por columna se derivan como `n_rows - non_null`, y el porcentaje se calcula contra el total de filas obtenido en la sección 3.

In [19]:
# El conteo se hace en dos pasos para mantener cada job ligero:
# (1) df.summary('count') sobre todas las columnas que soporta (numéricas
#     y string); este job ya se ejecutó sin problemas en 7.1.
# (2) F.count solo sobre las columnas de tipo timestamp_ntz, que summary
#     omite por defecto. Es una agregación de 2 columnas, muy ligera.
# Combinar 20 agregaciones en un solo job satura la heap por defecto, por
# eso se evita esa vía.
counts_pd = df.summary("count").toPandas().drop(columns=["summary"]).T
counts_pd.columns = ["non_null"]
counts_pd["non_null"] = counts_pd["non_null"].astype(int)

# Columnas de timestamp omitidas por summary: contar por separado.
ts_cols = [c for c, t in df.dtypes if t.startswith("timestamp") and c not in counts_pd.index]
if ts_cols:
    ts_row = df.agg(*[F.count(c).alias(c) for c in ts_cols]).first()
    for c in ts_cols:
        counts_pd.loc[c] = [ts_row[c]]

# Reordenar al orden original del DataFrame.
counts_pd = counts_pd.reindex(df.columns)

counts_pd["nulos"] = n_rows - counts_pd["non_null"]
counts_pd["pct"] = (counts_pd["nulos"] / n_rows * 100).round(2)
counts_pd.index.name = "columna"

print("Nulos por columna:")
print(counts_pd[["nulos", "pct"]].to_string())

Nulos por columna:
                          nulos    pct
columna                               
VendorID                      0   0.00
tpep_pickup_datetime          0   0.00
tpep_dropoff_datetime         0   0.00
passenger_count        15703126  17.47
trip_distance                 0   0.00
RatecodeID             15703126  17.47
store_and_fwd_flag     15703126  17.47
PULocationID                  0   0.00
DOLocationID                  0   0.00
payment_type                  0   0.00
fare_amount                   0   0.00
extra                         0   0.00
mta_tax                       0   0.00
tip_amount                    0   0.00
tolls_amount                  0   0.00
improvement_surcharge         0   0.00
total_amount                  0   0.00
congestion_surcharge   15703126  17.47
Airport_fee            15703126  17.47
cbd_congestion_fee     41169720  45.80


#### Interpretación del conteo de nulos

La tabla evidencia tres bloques de nulidad bien definidos:

- **Sin nulos (0% en el dataset completo)**: `VendorID`, `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `trip_distance`, `PULocationID`, `DOLocationID`, `payment_type`, `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge` y `total_amount`. Estas columnas conforman el núcleo operativo del registro de viaje y siempre se reportan.
- **Nulos correlacionados (15,703,126 filas, 17.47% del total)**: `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` y `Airport_fee` tienen exactamente el mismo número de nulos. Como se identificó en la sección 5.1, este bloque corresponde a viajes con `payment_type = 0` (Flex Fare): para estos viajes el vendedor no reporta los campos del taxímetro tradicionales.
- **Nulos por cobertura temporal (`cbd_congestion_fee`, 41,169,720 filas, 45.80% del total)**: la columna no existía en los archivos de 2024 ni en parte de los Flex Fare de 2025. No son datos faltantes sino ausencia esperada documentada en el diccionario oficial.

La etapa de calidad de datos decidirá la estrategia para los nulos del bloque Flex Fare según las necesidades de cada análisis posterior: imputar, mantener como categoría propia o filtrar.

### 7.3 Top-10 zonas de origen y destino

Los campos `PULocationID` y `DOLocationID` almacenan códigos numéricos del catálogo TLC. Un código como 237 no transmite información geográfica inmediata; en cambio, "Upper East Side North (Manhattan)" sí lo hace. Por eso se realiza un join con el DataFrame `zones` (caracterizado en la sección 3.1) para enriquecer los resultados con las columnas `Borough`, `Zone` y `service_zone`.

El join es eficiente porque `zones` tiene solo 265 filas y Spark lo convierte automáticamente en un broadcast join: la tabla pequeña se copia a cada executor, evitando el shuffle de los 90 millones de filas del DataFrame principal. El `groupBy().count()` se ejecuta antes del join para que el shuffle de agregación opere sobre el DataFrame grande antes de enriquecerlo.

In [20]:
# Top-10 zonas de origen (PULocationID)
top_pu = (
    df.groupBy("PULocationID")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
    .join(
        zones.select(
            F.col("LocationID").cast("smallint").alias("PULocationID"),
            "Borough",
            "Zone",
            "service_zone",
        ),
        on="PULocationID",
        how="left",
    )
    .orderBy(F.desc("count"))
)

print("Top-10 zonas de origen:")
top_pu.show(truncate=False)

# Top-10 zonas de destino (DOLocationID)
top_do = (
    df.groupBy("DOLocationID")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
    .join(
        zones.select(
            F.col("LocationID").cast("smallint").alias("DOLocationID"),
            "Borough",
            "Zone",
            "service_zone",
        ),
        on="DOLocationID",
        how="left",
    )
    .orderBy(F.desc("count"))
)

print("Top-10 zonas de destino:")
top_do.show(truncate=False)

Top-10 zonas de origen:
+------------+-------+---------+----------------------------+------------+
|PULocationID|count  |Borough  |Zone                        |service_zone|
+------------+-------+---------+----------------------------+------------+
|132         |4052450|Queens   |JFK Airport                 |Airports    |
|237         |4041001|Manhattan|Upper East Side South       |Yellow Zone |
|161         |4004361|Manhattan|Midtown Center              |Yellow Zone |
|236         |3604703|Manhattan|Upper East Side North       |Yellow Zone |
|162         |2927023|Manhattan|Midtown East                |Yellow Zone |
|230         |2907563|Manhattan|Times Sq/Theatre District   |Yellow Zone |
|186         |2905322|Manhattan|Penn Station/Madison Sq West|Yellow Zone |
|142         |2702304|Manhattan|Lincoln Square East         |Yellow Zone |
|138         |2586847|Queens   |LaGuardia Airport           |Airports    |
|170         |2470971|Manhattan|Murray Hill                 |Yellow Zone |
+

#### Interpretación de las zonas de mayor demanda

Los rankings observados revelan una asimetría operativa relevante entre orígenes y destinos:

- **Top-10 orígenes**: 8 de las 10 zonas son Yellow Zone de Manhattan (Upper East Side South, Midtown Center, Upper East Side North, Midtown East, Times Sq/Theatre District, Penn Station/Madison Sq West, Lincoln Square East, Murray Hill). Las dos restantes son aeropuertos: **JFK Airport ocupa el primer lugar absoluto con 4,052,450 viajes** y LaGuardia Airport ocupa el lugar 9 con 2,586,847 viajes.
- **Top-10 destinos**: las 10 zonas son exclusivamente Yellow Zone de Manhattan. Ningún aeropuerto aparece en el ranking.

La asimetría se explica por el modelo regulatorio TLC: en NYC, los taxis amarillos tienen acceso preferente a las filas de espera de pasajeros en aeropuertos (origen), mientras que los pasajeros que llegan a aeropuertos suelen llegar en vehículos privados o servicios de viaje compartido. Por eso los aeropuertos dominan los pickups del taxi amarillo y prácticamente desaparecen de los drop-offs.

Las zonas de Manhattan se superponen ampliamente entre ambos rankings: Upper East Side North y South, Midtown Center, Times Sq/Theatre District, Murray Hill, Midtown East y Lincoln Square East aparecen en ambos top-10. Estas zonas funcionan como hubs de circulación, tanto de salida como de llegada. Ningún registro del top-10 corresponde a los códigos 264 (Unknown) o 265 (Outside of NYC), lo que sugiere baja prevalencia de errores de georreferencia entre las zonas de mayor volumen, aunque no descarta su presencia en la cola de la distribución.

### 7.4 Patrones temporales

Para analizar patrones por hora del día, día de semana y mes, primero se deriva la columna `trip_duration_min` a partir de los timestamps de inicio y fin del viaje.

Los timestamps de NYC TLC están almacenados como `timestamp_ntz` (timestamp without timezone): valores de wall-clock sin información de zona horaria. Para calcular la duración entre dos `timestamp_ntz` de manera independiente de la zona horaria del entorno donde se ejecute el notebook, se usa `F.timestamp_diff(unit, start, end)`, función de PySpark 3.5+ que opera directamente sobre los componentes de wall-clock sin convertir a Unix seconds y, por lo tanto, no depende de la zona horaria de la sesión de Spark. Se solicita la diferencia en `SECOND` y se divide entre 60 para obtener minutos en formato decimal. Referencia: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.timestamp_diff.html

Las funciones de extracción temporal relevantes son:
- `F.hour(col)`: extrae la hora del día (0 a 23) desde un timestamp.
- `F.dayofweek(col)`: extrae el día de la semana como entero (1 = domingo, 2 = lunes, ..., 7 = sábado). La convención de Spark sigue el estándar ISO de Java donde 1 es domingo, no lunes.
- `F.month(col)`: extrae el mes (1 a 12).
- `F.year(col)`: extrae el año.

La columna `trip_duration_min` se agrega a `df` con `withColumn`, reasignando la referencia en sitio.

In [21]:
# Derivar trip_duration_min a partir de los timestamps.
# Se usa F.timestamp_diff, que opera directamente sobre wall-clock y no
# depende de la zona horaria de la sesión de Spark; esto importa porque
# el notebook puede ejecutarse en máquinas con cualquier TZ configurada.
# El primer argumento (unit) es un string Python, no una Column. La función
# devuelve un entero en la unidad solicitada (SECOND); se divide entre 60.0
# para obtener minutos como float.
df = df.withColumn(
    "trip_duration_min",
    F.timestamp_diff(
        "SECOND",
        F.col("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime"),
    ) / 60.0,
)

print("Columna trip_duration_min agregada.")
print(f"Total de columnas ahora: {len(df.columns)}")

Columna trip_duration_min agregada.
Total de columnas ahora: 21


In [22]:
# Patrón 1: distribución por hora del día (0-23).
# Muestra conteo, tarifa promedio y duración promedio por hora.
print("Distribución por hora del día:")
(
    df.groupBy(F.hour("tpep_pickup_datetime").alias("hora"))
    .agg(
        F.count("*").alias("viajes"),
        F.round(F.avg("fare_amount"), 2).alias("fare_promedio"),
        F.round(F.avg("trip_duration_min"), 2).alias("duracion_prom_min"),
    )
    .orderBy("hora")
    .show(24, truncate=False)
)

Distribución por hora del día:


+----+-------+-------------+-----------------+
|hora|viajes |fare_promedio|duracion_prom_min|
+----+-------+-------------+-----------------+
|0   |2732251|18.98        |15.24            |
|1   |1790778|17.32        |14.06            |
|2   |1172559|16.16        |13.26            |
|3   |788864 |16.88        |13.26            |
|4   |604740 |21.55        |14.95            |
|5   |649845 |24.41        |16.99            |
|6   |1345200|21.12        |17.43            |
|7   |2517058|18.94        |17.1             |
|8   |3431869|18.23        |16.99            |
|9   |3707953|17.99        |16.95            |
|10  |3914840|18.23        |17.34            |
|11  |4225716|18.5         |17.92            |
|12  |4613962|19.04        |18.29            |
|13  |4820757|19.4         |18.78            |
|14  |5173535|20.24        |19.93            |
|15  |5366201|20.18        |20.38            |
|16  |5496380|20.33        |20.35            |
|17  |6104414|18.97        |18.97            |
|18  |6428945

In [23]:
# Patrón 2: distribución por día de semana.
# En Spark, F.dayofweek devuelve 1=domingo, 2=lunes, ..., 7=sábado.
# Se agrega una columna de etiqueta para claridad.
dia_labels = {1: "domingo", 2: "lunes", 3: "martes", 4: "miércoles",
              5: "jueves", 6: "viernes", 7: "sábado"}

print("Distribución por día de semana (1=domingo ... 7=sabado en Spark):")
(
    df.groupBy(F.dayofweek("tpep_pickup_datetime").alias("dia_semana"))
    .agg(
        F.count("*").alias("viajes"),
        F.round(F.avg("fare_amount"), 2).alias("fare_promedio"),
    )
    .orderBy("dia_semana")
    .show(7, truncate=False)
)

Distribución por día de semana (1=domingo ... 7=sabado en Spark):


+----------+--------+-------------+
|dia_semana|viajes  |fare_promedio|
+----------+--------+-------------+
|1         |11726699|19.56        |
|2         |11063437|19.51        |
|3         |12661161|18.64        |
|4         |13410382|18.71        |
|5         |13905489|18.99        |
|6         |13357499|18.66        |
|7         |13767655|17.81        |
+----------+--------+-------------+



In [24]:
# Patrón 3: conteo mensual cruzado con año (2024 y 2025).
# Produce hasta 24 filas (12 meses x 2 años). Los años fuera de rango
# (2002, 2007-2009, 2023, 2026) observados en sección 5.3 aparecerán
# pero con conteos mínimos; se documentan como problema de calidad.
print("Conteo de viajes por año y mes:")
(
    df.groupBy(
        F.year("tpep_pickup_datetime").alias("año"),
        F.month("tpep_pickup_datetime").alias("mes"),
    )
    .agg(F.count("*").alias("viajes"))
    .orderBy("año", "mes")
    .show(30, truncate=False)
)

Conteo de viajes por año y mes:


+----+---+-------+
|año |mes|viajes |
+----+---+-------+
|2002|12 |11     |
|2007|12 |1      |
|2008|12 |11     |
|2009|1  |24     |
|2023|12 |10     |
|2024|1  |2964617|
|2024|2  |3007533|
|2024|3  |3582611|
|2024|4  |3514295|
|2024|5  |3723843|
|2024|6  |3539170|
|2024|7  |3076876|
|2024|8  |2979192|
|2024|9  |3633025|
|2024|10 |3833780|
|2024|11 |3646372|
|2024|12 |3668371|
|2025|1  |3475236|
|2025|2  |3577543|
|2025|3  |4145231|
|2025|4  |3970568|
|2025|5  |4591844|
|2025|6  |4322949|
|2025|7  |3898971|
|2025|8  |3574080|
|2025|9  |4251019|
|2025|10 |4428708|
|2025|11 |4181432|
|2025|12 |4304997|
|2026|6  |2      |
+----+---+-------+



#### Interpretación de los patrones temporales

**Por hora del día**: el volumen describe una curva diurna clara. El valle nocturno se concentra entre las 3 y 5 AM (mínimo absoluto a las 4 AM con 604,740 viajes); a partir de las 5 AM el volumen crece de manera sostenida y alcanza el máximo absoluto a las 18:00 (6,428,945 viajes). Las horas de la tarde-noche (15:00 a 22:00) concentran el mayor volumen diario. La tarifa promedio máxima ocurre a las 5 AM (USD 24.41), seguida por las 4 AM (USD 21.55): los pocos viajes de madrugada tienden a ser largos y de tarifa alta. La duración promedio del viaje es máxima entre las 14:00 y 16:00 (19.93 a 20.38 minutos), reflejo del tráfico vespertino.

**Por día de semana** (1 = domingo, 7 = sábado en la convención de Spark): el día de mayor volumen es **jueves (5)** con 13,905,489 viajes, seguido por **sábado (7)** con 13,767,655. El día de menor volumen es **lunes (2)** con 11,063,437 viajes; domingo se ubica apenas por encima con 11,726,699. La tarifa promedio más alta se observa en **domingo (USD 19.56)** y **lunes (USD 19.51)**, mientras que **sábado registra la tarifa promedio más baja (USD 17.81)**. Lectura operativa: durante la semana laboral los viajes son rutinarios y predominantemente intra-Manhattan; el sábado, a pesar del alto volumen, baja la tarifa promedio porque los viajes son más cortos y de entretenimiento dentro de la ciudad.

**Por mes y año**: 2024 promedia ~3.4M viajes mensuales (rango 2.96M en enero a 3.83M en octubre); 2025 promedia ~4.0M (rango 3.48M en enero a 4.59M en mayo). El crecimiento interanual ronda 15-20% en cada mes comparable. En ambos años hay caídas en enero y agosto, consistentes con periodos de menor actividad turística y vacaciones en NYC. La tabla incluye además 59 registros con timestamps fuera del rango 2024-2025 (años 2002, 2007, 2008, 2009, 2023 y 2026), que se documentan como problema de calidad temporal a resolver en la etapa siguiente.

### 7.5 Matriz de correlación

Spark MLlib no puede calcular correlaciones directamente sobre columnas sueltas de un DataFrame: sus algoritmos estadísticos y de machine learning esperan que todas las variables de entrada estén empaquetadas en una sola columna de tipo `Vector`. `VectorAssembler` es el transformador de la librería `pyspark.ml.feature` que realiza esta conversión: toma una lista de columnas numéricas (`inputCols`) y genera una nueva columna de tipo `DenseVector` o `SparseVector` (`outputCol`).

El parámetro `handleInvalid="skip"` indica al ensamblador que descarte silenciosamente cualquier fila que contenga un valor nulo en alguna de las columnas de entrada. Esto es necesario aquí porque `cbd_congestion_fee` es nulo para todos los registros de 2024 (aproximadamente 41.2 millones de filas, el 46% del dataset), y otros campos del bloque Flex Fare también presentan nulos. La alternativa `handleInvalid="error"` lanzaría una excepción al encontrar el primer nulo, y `"keep"` sustituiría los nulos por cero, lo que distorsionaría las correlaciones.

Consecuencia importante: la matriz de correlación se calcula sobre el subconjunto de filas sin ningún nulo en las 13 columnas ensambladas. Dado que `cbd_congestion_fee` es nulo en todo 2024, la matriz refleja esencialmente los viajes de 2025 más los viajes no-Flex-Fare de 2024. Este sesgo temporal se debe tener en cuenta al interpretar los coeficientes.

Una vez ensamblado el vector, `Correlation.corr(df_assembled, "features", "pearson")` calcula la matriz de correlación de Pearson en forma distribuida y devuelve un DataFrame de una sola fila con la matriz como `DenseMatrix`. Se convierte a pandas para visualización.

Referencias:
- https://spark.apache.org/docs/latest/ml-statistics.html (sección Correlation)
- https://spark.apache.org/docs/latest/ml-features.html#vectorassembler

In [25]:
import pandas as pd
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# Columnas a incluir en la matriz de correlación.
# Se incluyen todas las numéricas continuas más trip_duration_min.
# Se excluyen columnas categóricas codificadas (VendorID, RatecodeID,
# payment_type, PULocationID, DOLocationID) y los timestamps.
corr_cols = [
    "passenger_count",
    "trip_distance",
    "trip_duration_min",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
]

# VectorAssembler: empaqueta las columnas en una sola columna "features".
# handleInvalid="skip" descarta filas con cualquier nulo en las columnas
# ensambladas. Esto excluye ~46% de las filas por cbd_congestion_fee (2024)
# y adicionalmente las filas Flex Fare con nulos en otros campos.
assembler = VectorAssembler(
    inputCols=corr_cols,
    outputCol="features",
    handleInvalid="skip",
)

df_assembled = assembler.transform(df).select("features")

# Cálculo de la matriz de correlación de Pearson.
# El resultado es un DataFrame de 1 fila con la DenseMatrix.
corr_matrix_row = Correlation.corr(df_assembled, "features", "pearson").head()
corr_matrix = corr_matrix_row[0].toArray()

# Convertir a pandas para visualización tabular.
corr_pd = pd.DataFrame(corr_matrix, index=corr_cols, columns=corr_cols).round(2)

print(f"Matriz de correlación de Pearson ({len(corr_cols)}x{len(corr_cols)})")
print(f"Nota: calculada sobre filas sin nulos en ninguna de las {len(corr_cols)} columnas.")
print()
print(corr_pd.to_string())

26/05/03 02:29:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Matriz de correlación de Pearson (13x13)
Nota: calculada sobre filas sin nulos en ninguna de las 13 columnas.

                       passenger_count  trip_distance  trip_duration_min  fare_amount  extra  mta_tax  tip_amount  tolls_amount  improvement_surcharge  total_amount  congestion_surcharge  Airport_fee  cbd_congestion_fee
passenger_count                   1.00           0.00               0.02         0.01  -0.06    -0.01        0.03          0.03                   0.01          0.01                 -0.00         0.03                0.02
trip_distance                     0.00           1.00               0.03         0.01   0.01    -0.00        0.03          0.04                  -0.01          0.01                 -0.02         0.04               -0.01
trip_duration_min                 0.02           0.03               1.00         0.04   0.04    -0.00        0.20          0.20                  -0.07          0.05                 -0.10         0.18               -0.01
fare_amou

#### Interpretación de la matriz de correlación

La matriz observada **revela un hallazgo metodológico crítico**: las correlaciones que estructuralmente deberían ser altas son cercanas a cero, mientras emergen agrupaciones inesperadas. Esto se explica por la presencia de outliers extremos en `trip_distance` (max 398,608 millas) y en variables monetarias (`fare_amount` max USD 863,372, `total_amount` max USD 863,380): el coeficiente de Pearson es muy sensible a outliers, y un puñado de valores absurdos infla la varianza al punto de enmascarar las relaciones lineales reales.

Correlaciones que deberían ser altas y se ven distorsionadas por outliers:

- `trip_distance` vs `fare_amount`: r = 0.01. Estructuralmente la tarifa se calcula en función de la distancia recorrida; en datos limpios el coeficiente debería superar 0.8.
- `trip_distance` vs `trip_duration_min`: r = 0.03. A mayor distancia mayor duración, salvo por el ruido del tráfico urbano; el r observado refleja la misma distorsión por outliers.
- `fare_amount` vs `tip_amount`: r = 0.07. La propina suele ser un porcentaje de la tarifa para pagos con tarjeta; los outliers la disipan.
- `fare_amount` vs `total_amount`: r = 1.00. Esta sí se sostiene porque `total_amount` es la suma de `fare_amount` más recargos casi fijos, y los outliers afectan ambas variables de forma proporcional.

Correlaciones moderadas-altas que sí emergen incluso sobre datos crudos:

- **Cluster de recargos por zona de congestión**: `improvement_surcharge` vs `congestion_surcharge` r = 0.74; `congestion_surcharge` vs `cbd_congestion_fee` r = 0.60; `improvement_surcharge` vs `cbd_congestion_fee` r = 0.47. Indica que los viajes dentro de la zona de descongestión central de Manhattan acumulan los tres recargos juntos.
- **Cluster de viajes al aeropuerto**: `Airport_fee` vs `tolls_amount` r = 0.45; `Airport_fee` vs `tip_amount` r = 0.41; `tolls_amount` vs `tip_amount` r = 0.45. Los viajes a/desde aeropuertos pasan por peajes, generan tarifas más altas y atraen propinas más generosas.
- **Relación recargos extra**: `extra` vs `Airport_fee` r = 0.30; `extra` vs `improvement_surcharge` r = 0.23; `extra` vs `tolls_amount` r = 0.22. El cargo `extra` (recargos misceláneos) se asocia con los demás recargos del viaje.

Conclusión: esta matriz no es válida como referencia de las relaciones genuinas entre variables; es prueba directa de que la limpieza de outliers en la etapa siguiente es indispensable antes de cualquier modelado supervisado o no supervisado. Una matriz post-limpieza, o el uso del coeficiente de Spearman (basado en rangos y robusto a outliers), debería arrojar valores muy distintos para las correlaciones estructurales esperadas (`trip_distance` vs `fare_amount`, `trip_distance` vs `trip_duration_min`, `fare_amount` vs `tip_amount`).

### Cierre de la sección 7

Los agregados calculados en esta sección quedan disponibles para los pasos siguientes:

- La etapa de visualización utilizará los resultados de `summary()`, las tablas por hora, día y mes, y la matriz de correlaciones para generar gráficos de distribución, mapas de calor y series de tiempo.
- La etapa de calidad de datos partirá de la tabla de nulos, los rangos extremos del resumen estadístico y la presencia de los IDs 264 y 265 en el top de zonas como insumos para definir las reglas de limpieza.